In [2]:
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
import os
import pickle

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
def load_and_merge_embeddings(folder_path):
    features_dict = {}

    #ключи должны соответствовать значениям в колонках 'type_entity' dataframe
    mapping = {
        "DNA": "dnabert_DNA.pkl",
        "NucleicAmbigous": "dnabert_NucleicAmbigous.pkl",
        "NucleicMixed": "dnabert_NucleicMixed.pkl",
        "AA": "protein_esm_35M.pkl",
        "RNA": "rna_berta.pkl",
        "SmallMolecule": "sm_chemberta_10M_MTR.pkl"
    }

    for entity_type, file_name in mapping.items():
        file_path = os.path.join(folder_path, file_name)

        if os.path.exists(file_path):
            with open(file_path, 'rb') as f:
                # Загружаем словарь {raw_id: embedding}
                data = pickle.load(f)
                features_dict[entity_type] = data
                print(f"Loaded {entity_type}: {len(data)} entities")
        else:
            print(f"Warning: File {file_name} not found in {folder_path}")

    return features_dict

In [22]:
def df_to_homo_two_rel(df: pd.DataFrame, features_dict: dict):
    #получаем df: node_type->raw_id
    nodes = pd.concat([
        df[["type_entity_1","id_entity_1"]].rename(columns={"type_entity_1":"t","id_entity_1":"id"}),
        df[["type_entity_2","id_entity_2"]].rename(columns={"type_entity_2":"t","id_entity_2":"id"}),
    ], axis=0).drop_duplicates()

    key = list(zip(nodes["t"].astype(str), nodes["id"].astype(int))) #cписок из кортежей (node_type, raw_id)
    node_map = {k:i for i,k in enumerate(key)} #словарь вида {(node_type, raw_id):id_nodes}
    num_nodes = len(node_map) #общее количество узлов

    src = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_1"], df["id_entity_1"])] #спиcок id_nodes для heads триплетов
    dst = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_2"], df["id_entity_2"])] #спиcок id_nodes для tales триплетов
    edge_index = torch.tensor([src, dst], dtype=torch.long) #edge_index графа (все ребра) в виде tensor[2,num_edges]

    preds = df["predicate"].astype(str).unique().tolist() #список всех предикатов в графе
    pred2pred_id = {p:i for i,p in enumerate(sorted(preds))} #словарь {pred:pred_id}
    edge_type = torch.tensor([pred2pred_id[p] for p in df["predicate"].astype(str)], dtype=torch.long) #pred_id ребер графа в виде tensor[num_edges]

    edge_index, edge_type = to_undirected(edge_index, edge_type) #дублируем напарвления т.к. граф неориентированный

    #создание словаря вида {node_type:raw_emb_size}
    dims_dict = {}
    for t, emb_dict in features_dict.items():
        sample_emb = next(iter(emb_dict.values()))
        dims_dict[t] = len(sample_emb)

    unique_types = list(features_dict.keys()) #все представленный типы узлов
    type2id = {t: i for i, t in enumerate(unique_types)} #словарь {node_type:node_type_id}

    max_dim = max(dims_dict.values())  #размерность максимального эмбеддинга
    x = torch.zeros(num_nodes, max_dim) #в дальнейшем заполним эмбеддингами
    node_type_tensor = torch.zeros(num_nodes, dtype=torch.long) #сразу создаем пустой тензор

    #итерируемся напрямую по маппингу
    missing_features = 0
    for (t, raw_id), mapped_id in node_map.items():
        dim = dims_dict[t] #размерность эмбеддинга текущей вершины

        #получаем эмбеддинг
        if raw_id in features_dict[t]:
            emb = features_dict[t][raw_id]
            if not isinstance(emb, torch.Tensor):
                emb = torch.tensor(emb, dtype=torch.float)
        else:
            emb = torch.zeros(dim)
            missing_features += 1

        #записываем в x наш эмбеддинг
        x[mapped_id, :dim] = emb
        node_type_tensor[mapped_id] = type2id[t]
    #на случай ошибок
    if missing_features > 0:
        print(f"Warning: {missing_features} nodes are missing features and were zero-padded.")

    #созадем torch-geomertic data
    data = Data(x=x, edge_index=edge_index)
    data.edge_type = edge_type
    data.node_type = node_type_tensor

    return data, node_map, pred2pred_id, type2id, dims_dict

In [4]:
df = pd.read_csv('../data/edges/clean_triples.csv')

In [11]:
folder = "../data/dicts/bert_embeddings"
all_features = load_and_merge_embeddings(folder)

Loaded DNA: 630 entities
Loaded NucleicAmbigous: 16 entities
Loaded NucleicMixed: 23 entities
Loaded AA: 71376 entities
Loaded RNA: 749 entities
Loaded SmallMolecule: 1301937 entities


In [8]:
nodes = pd.concat([
        df[["type_entity_1","id_entity_1"]].rename(columns={"type_entity_1":"t","id_entity_1":"id"}),
        df[["type_entity_2","id_entity_2"]].rename(columns={"type_entity_2":"t","id_entity_2":"id"}),
    ], axis=0).drop_duplicates()

key = list(zip(nodes["t"].astype(str), nodes["id"].astype(int))) #cписок из кортежей (node_type, raw_id)
node_map = {k:i for i,k in enumerate(key)} #словарь вида {(node_type, raw_id):id_nodes}
num_nodes = len(node_map) #общее количество узлов

In [12]:
src = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_1"], df["id_entity_1"])]
dst = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_2"], df["id_entity_2"])]
edge_index = torch.tensor([src, dst], dtype=torch.long)

In [18]:
preds = df["predicate"].astype(str).unique().tolist()
pred2pred_id = {p:i for i,p in enumerate(sorted(preds))}
edge_type = torch.tensor([pred2pred_id[p] for p in df["predicate"].astype(str)], dtype=torch.long)

tensor([1, 1, 1,  ..., 0, 0, 0])

In [12]:
unique_types = list(all_features.keys()) #все представленный типы узлов
type2id = {t: i for i, t in enumerate(unique_types)}
type2id

{'DNA': 0,
 'NucleicAmbigous': 1,
 'NucleicMixed': 2,
 'AA': 3,
 'RNA': 4,
 'SmallMolecule': 5}

In [15]:
node_types_list = nodes["t"].astype(str).tolist()
node_type_tensor = torch.tensor([type2id[t] for t in node_types_list], dtype=torch.long)


In [16]:
node_ids_list = nodes["id"].astype(int).tolist()

for i, (t, raw_id) in enumerate(zip(node_types_list, node_ids_list)):
    dim = dims_dict[t]
    emb = features_dict[t].get(raw_id, torch.zeros(dim))

[72694,
 71625,
 71753,
 72701,
 72450,
 71683,
 72140,
 72290,
 72054,
 72505,
 71635,
 72672,
 71837,
 72720,
 72069,
 71928,
 1374712,
 1374717,
 71923,
 71804,
 71629,
 71429,
 72297,
 71693,
 72174,
 72460,
 72068,
 72187,
 71834,
 71430,
 71682,
 71517,
 72356,
 71953,
 72152,
 72020,
 72458,
 1374718,
 71392,
 71954,
 72260,
 72086,
 72727,
 1374713,
 71543,
 72120,
 72396,
 71871,
 71479,
 72267,
 71428,
 72151,
 71920,
 71466,
 1374695,
 71686,
 72277,
 71938,
 72704,
 72355,
 72133,
 72111,
 71620,
 72429,
 71805,
 71656,
 71818,
 72432,
 71912,
 71600,
 72749,
 71636,
 71465,
 72352,
 71507,
 71431,
 72253,
 72075,
 72353,
 72319,
 72170,
 71794,
 72615,
 72457,
 72074,
 71916,
 71657,
 72509,
 72163,
 71541,
 71816,
 72693,
 72226,
 71641,
 72191,
 71793,
 72165,
 72257,
 72446,
 72677,
 71614,
 72079,
 71506,
 72272,
 72158,
 71915,
 71940,
 71410,
 71403,
 72169,
 72315,
 72581,
 71903,
 71735,
 72049,
 72695,
 71552,
 72156,
 72611,
 72459,
 71419,
 72159,
 72685,
 71918